### Revenue and Units sold by region

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW mavenmarket.gold.mv_revenue_by_region 
SCHEDULE REFRESH EVERY 24 HOURS
COMMENT 'Geographic sales distribution refreshed automatically every 6 hours'
AS
SELECT 
    r.sales_region_name,
    r.sales_district_name,
    d.year_number,
    d.month_number,
    d.month_name,
    COUNT(DISTINCT fs.sales_fact_sk) AS total_transaction_lines,
    COUNT(DISTINCT fs.customer_sk) AS unique_customers,
    SUM(fs.sales_quantity) AS total_units_sold,
    SUM(fs.gross_revenue_amount) AS total_gross_revenue,
    SUM(fs.total_cost_amount) AS total_cost,
    SUM(fs.gross_profit_amount) AS total_gross_profit
FROM mavenmarket.gold.fact_sales fs
JOIN mavenmarket.gold.dim_store s 
    ON fs.store_sk = s.store_sk
JOIN mavenmarket.gold.dim_region r 
    ON s.region_id = r.region_id  -- (or s.region_sk = r.region_sk)
JOIN mavenmarket.gold.dim_date d 
    ON fs.transaction_date_sk = d.date_sk
GROUP BY 
    r.sales_region_name,
    r.sales_district_name,
    d.year_number,
    d.month_number,
    d.month_name;

In [0]:
%sql
select * from mavenmarket.gold.mv_revenue_by_region

### Returns Percentage & Performance by Month

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW mavenmarket.gold.mv_returns_pct_by_month 
SCHEDULE REFRESH EVERY 24 HOURS
COMMENT 'Monthly product return percentage and store return metrics'
AS
WITH monthly_sales AS (
    SELECT 
        d.year_number,
        d.month_number,
        d.month_name,
        fs.product_sk,
        fs.store_sk,
        SUM(fs.sales_quantity) AS total_units_sold
    FROM mavenmarket.gold.fact_sales fs
    JOIN mavenmarket.gold.dim_date d 
        ON fs.transaction_date_sk = d.date_sk
    GROUP BY d.year_number, d.month_number, d.month_name, fs.product_sk, fs.store_sk
),
monthly_returns AS (
    SELECT 
        d.year_number,
        d.month_number,
        fr.product_sk,
        fr.store_sk,
        SUM(fr.return_quantity) AS total_units_returned
    FROM mavenmarket.gold.fact_returns fr
    JOIN mavenmarket.gold.dim_date d 
        ON fr.return_date_key = d.date_sk
    GROUP BY d.year_number, d.month_number, fr.product_sk, fr.store_sk
)
SELECT 
    ms.year_number,
    ms.month_number,
    ms.month_name,
    s.store_name,
    s.city_name,
    s.state_province_code,
    SUM(ms.total_units_sold) AS gross_units_sold,
    COALESCE(SUM(mr.total_units_returned), 0) AS units_returned,
    ROUND(
        COALESCE(SUM(mr.total_units_returned), 0) * 100.0 / NULLIF(SUM(ms.total_units_sold), 0), 
        2
    ) AS return_rate_percentage
FROM monthly_sales ms
LEFT JOIN monthly_returns mr 
    ON ms.year_number = mr.year_number 
   AND ms.month_number = mr.month_number 
   AND ms.product_sk = mr.product_sk 
   AND ms.store_sk = mr.store_sk
JOIN mavenmarket.gold.dim_store s 
    ON ms.store_sk = s.store_sk
GROUP BY 
    ms.year_number,
    ms.month_number,
    ms.month_name,
    s.store_name,
    s.city_name,
    s.state_province_code;

In [0]:
%sql
select * from mavenmarket.gold.mv_returns_pct_by_month 

### Profit by Category

In [0]:

%sql
CREATE OR REPLACE MATERIALIZED VIEW mavenmarket.gold.mv_profit_by_category 
SCHEDULE REFRESH EVERY 24 HOURS
COMMENT 'Product profit margin performance grouped by category and brand per quarter'
AS
SELECT 
    p.category_name,
    p.brand_name,
    d.year_number,
    d.quarter_number,
    SUM(s.gross_revenue_amount) AS total_revenue_amount,
    SUM(s.total_cost_amount) AS total_cost_amount,
    SUM(s.gross_profit_amount) AS total_gross_profit_amount,
    ROUND((SUM(s.gross_profit_amount) / NULLIF(SUM(s.gross_revenue_amount), 0)) * 100, 2) AS profit_margin_percentage
FROM mavenmarket.gold.fact_sales s
JOIN mavenmarket.gold.dim_product p 
    ON s.product_sk = p.product_sk
JOIN mavenmarket.gold.dim_date d 
    ON s.transaction_date_sk = d.date_sk
GROUP BY 
    p.category_name,
    p.brand_name,
    d.year_number,
    d.quarter_number
ORDER BY 
    total_gross_profit_amount DESC;

In [0]:
%sql
SELECT * 
FROM mavenmarket.gold.mv_profit_by_category
ORDER BY total_gross_profit_amount DESC;